<a href="https://colab.research.google.com/github/kartikeyabekkari/HyperRealisticNisqSimulation/blob/main/Hyper_Realistic_NISQ_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup & Imports



In [ ]:
!pip install --upgrade --force-reinstall cirq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.7/733.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.

In [ ]:
import warnings
warnings.filterwarnings("ignore")


try:
    import cirq
except ImportError:
    print("installing cirq...")
    !pip install cirq --quiet
    import cirq
    print("installed cirq.")


import matplotlib.pyplot as plt
import cirq_web
import numpy as np

print("Libraries Imported Successfully!")

Libraries Imported Successfully!




---


#Initialization


---



In [5]:
#================ Parameters =================

number_of_qubits_in_the_circuit = 2
number_of_runs = 8000
base_noise = 0.01 # <---- In Decimal Representation of Percentage
high_noise = 0.027 # <---- In Decimal Representation of Percentage
distance_between_qubits = 2 # <---- In Micro Meters
frequency_overlap = 0 # <---- In Gigahertz

#=============== Build Circuit ===============
qubits = cirq.LineQubit.range(number_of_qubits_in_the_circuit)
base_circuit = cirq.Circuit([

#---------------------------------------------
cirq.H(qubits[0]),
cirq.CNOT(qubits[0], qubits[1]),
cirq.measure(qubits, key='result')
#---------------- Edit Here ^ ----------------

])
print('Your Ideal Circuit:\n')
print(f'{base_circuit}\n')
#=============================================


Your Ideal Circuit:

0: ───H───@───M('result')───
          │   │
1: ───────X───M─────────────



#Functions

In [6]:
def inject_errors(original_circuit):
      noisy_circuit = cirq.Circuit()
      for moment in original_circuit:
        # Add the original operations first
          noisy_circuit.append(moment)

        # Skip noise injection entirely if moment contains a measurement
          if any(isinstance(op.gate, cirq.MeasurementGate) for op in moment):
              continue

          active_loud_qubits = set()

          for op in moment:
              is_loud_gate = isinstance(op.gate, (
                  cirq.XPowGate,
                  cirq.YPowGate,
                  cirq.ZPowGate,
                  cirq.HPowGate,
                  cirq.CXPowGate,
                  cirq.PhasedXPowGate
              ))

              if is_loud_gate:
                  active_loud_qubits.update(op.qubits)

          error_gates = []
          for q in original_circuit.all_qubits():
              # Sets error to
              error_rate = high_noise if q in active_loud_qubits else base_noise
              error_gates.append(cirq.bit_flip(p=error_rate)(q))

          # Add the error gates to the noisy circuit
          noisy_circuit.append(cirq.Moment(error_gates))

      return noisy_circuit
inject_errors(base_circuit)

0: ───H───BF(0.027)───@───BF(0.027)───M('result')───
                      │               │
1: ───────BF(0.01)────X───BF(0.027)───M─────────────

In [7]:
def find_errors(circuit):
  simulator = cirq.Simulator()
    # Store list of moments directly to check what gates are inside
  noisy_moments = list(circuit)
  error_qubit_stream = []

    # Stochastic trajectory sampling replaces the original broken .sample() step here
  for current_moment in noisy_moments:
        is_error_moment = any(isinstance(op.gate, cirq.BitFlipChannel) for op in current_moment)
        if is_error_moment:
            flipped_qubits = []
            for op in current_moment:
                if isinstance(op.gate, cirq.BitFlipChannel):
                    p_val = op.gate.p
                    # FIX: Access the single qubit from the op.qubits tuple
                    q_idx = qubits.index(op.qubits[0])
                    if np.random.rand() < p_val:
                        flipped_qubits.append(q_idx)
            if flipped_qubits:
                error_qubit_stream.append(flipped_qubits)
            else:
                error_qubit_stream.append(0)
        else:
            error_qubit_stream.append(0)
find_errors(inject_errors(base_circuit))

In [ ]:
print("AHHHHHH")

AHHHHHH
